In [8]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [9]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [ ]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "202301"
END_YYYYMM = "202507"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 9 (202508 -> 202604) newest-first
[3/9] fetch EUR/USD 202602


/tmp/ipykernel_10388/3175562053.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  END_YYYYMM = datetime.utcnow().strftime("%Y%m")
INFO:DUKASCRIPT:current timestamp :2026-02-02T05:03:19.717000
INFO:DUKASCRIPT:current timestamp :2026-02-02T10:41:08.710000
INFO:DUKASCRIPT:current timestamp :2026-02-02T15:31:06.112000
INFO:DUKASCRIPT:current timestamp :2026-02-02T22:28:23.130000
INFO:DUKASCRIPT:current timestamp :2026-02-03T10:05:53.611000
INFO:DUKASCRIPT:current timestamp :2026-02-03T16:53:54.095000
INFO:DUKASCRIPT:current timestamp :2026-02-04T05:01:51.388000
INFO:DUKASCRIPT:current timestamp :2026-02-04T14:31:19.420000
INFO:DUKASCRIPT:current timestamp :2026-02-04T19:38:54.208000
INFO:DUKASCRIPT:current timestamp :2026-02-05T09:41:03.259000
INFO:DUKASCRIPT:current timestamp :2026-02-05T15:01:06.556000
INFO:DUKASCRIPT:current tim

  Received 1,330,089 ticks
Synced repo -> Drive: 6 new file(s)
[4/9] fetch EUR/USD 202601


INFO:DUKASCRIPT:current timestamp :2026-01-02T13:19:00.651000
INFO:DUKASCRIPT:current timestamp :2026-01-05T00:54:17.147000
INFO:DUKASCRIPT:current timestamp :2026-01-05T12:23:58.650000
INFO:DUKASCRIPT:current timestamp :2026-01-05T20:09:21.998000
INFO:DUKASCRIPT:current timestamp :2026-01-06T12:33:10.452000
INFO:DUKASCRIPT:current timestamp :2026-01-06T20:43:00.058000
INFO:DUKASCRIPT:current timestamp :2026-01-07T13:08:38.378000
INFO:DUKASCRIPT:current timestamp :2026-01-07T23:31:18.502000
INFO:DUKASCRIPT:current timestamp :2026-01-08T15:54:03.348000
INFO:DUKASCRIPT:current timestamp :2026-01-09T11:34:47.491000
INFO:DUKASCRIPT:current timestamp :2026-01-09T16:49:28.127000
INFO:DUKASCRIPT:current timestamp :2026-01-12T08:55:20.399000
INFO:DUKASCRIPT:current timestamp :2026-01-12T18:55:39.132000
INFO:DUKASCRIPT:current timestamp :2026-01-13T13:22:13.965000
INFO:DUKASCRIPT:current timestamp :2026-01-13T23:40:27.352000
INFO:DUKASCRIPT:current timestamp :2026-01-14T14:33:01.073000
INFO:DUK

  Received 1,506,261 ticks
Synced repo -> Drive: 2 new file(s)
[5/9] fetch EUR/USD 202512


...

  Received 1,486,651 ticks
Synced repo -> Drive: 2 new file(s)
[8/9] fetch EUR/USD 202509


INFO:DUKASCRIPT:current timestamp :2025-09-01T15:18:26.040000
INFO:DUKASCRIPT:current timestamp :2025-09-02T08:08:21.543000
INFO:DUKASCRIPT:current timestamp :2025-09-02T13:02:21.256000
INFO:DUKASCRIPT:current timestamp :2025-09-02T20:56:01.168000
INFO:DUKASCRIPT:current timestamp :2025-09-03T09:21:40.443000
INFO:DUKASCRIPT:current timestamp :2025-09-03T16:13:10.725000
INFO:DUKASCRIPT:current timestamp :2025-09-04T09:01:06.316000
INFO:DUKASCRIPT:current timestamp :2025-09-04T16:07:45.425000
INFO:DUKASCRIPT:current timestamp :2025-09-05T11:39:08.037000
INFO:DUKASCRIPT:current timestamp :2025-09-05T14:30:40.143000
INFO:DUKASCRIPT:current timestamp :2025-09-08T02:33:42.442000
INFO:DUKASCRIPT:current timestamp :2025-09-08T14:32:42.323000
INFO:DUKASCRIPT:current timestamp :2025-09-09T07:37:38.184000
INFO:DUKASCRIPT:current timestamp :2025-09-09T14:20:52.021000
INFO:DUKASCRIPT:current timestamp :2025-09-10T00:22:08.720000
INFO:DUKASCRIPT:current timestamp :2025-09-10T12:30:47.029000
INFO:DUK

  Received 1,493,839 ticks
Synced repo -> Drive: 2 new file(s)
[9/9] fetch EUR/USD 202508


INFO:DUKASCRIPT:current timestamp :2025-08-01T12:29:56.738000
INFO:DUKASCRIPT:current timestamp :2025-08-01T13:46:23.527000
INFO:DUKASCRIPT:current timestamp :2025-08-01T16:58:10.966000
INFO:DUKASCRIPT:current timestamp :2025-08-04T01:36:10.184000
INFO:DUKASCRIPT:current timestamp :2025-08-04T10:52:54.834000
INFO:DUKASCRIPT:current timestamp :2025-08-04T18:24:01.040000
INFO:DUKASCRIPT:current timestamp :2025-08-05T08:22:20.129000
INFO:DUKASCRIPT:current timestamp :2025-08-05T15:06:11.843000
INFO:DUKASCRIPT:current timestamp :2025-08-06T09:05:35.835000
INFO:DUKASCRIPT:current timestamp :2025-08-06T16:50:18.681000
INFO:DUKASCRIPT:current timestamp :2025-08-07T08:12:59.683000
INFO:DUKASCRIPT:current timestamp :2025-08-07T14:10:30.050000
INFO:DUKASCRIPT:current timestamp :2025-08-08T00:27:52.770000
INFO:DUKASCRIPT:current timestamp :2025-08-08T12:16:00.444000
INFO:DUKASCRIPT:current timestamp :2025-08-11T03:04:28.651000
INFO:DUKASCRIPT:current timestamp :2025-08-11T13:51:22.979000
INFO:DUK

  Received 1,341,676 ticks
Synced repo -> Drive: 2 new file(s)
Synced repo -> Drive: 0 new file(s)

Done
Downloaded: 7
Skipped existing: 2
Failed: 0
Newest downloaded months: ['202602', '202601', '202512', '202511', '202510', '202509', '202508']


In [11]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 22
Total size: 284.9 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_202101.parquet,202101,ask,18.387604
11,eurusd_dukascopy_bid_202101.parquet,202101,bid,18.384328
1,eurusd_dukascopy_ask_202102.parquet,202102,ask,15.310751
12,eurusd_dukascopy_bid_202102.parquet,202102,bid,15.227004
2,eurusd_dukascopy_ask_202508.parquet,202508,ask,12.352547
13,eurusd_dukascopy_bid_202508.parquet,202508,bid,12.384845
3,eurusd_dukascopy_ask_202509.parquet,202509,ask,13.929453
14,eurusd_dukascopy_bid_202509.parquet,202509,bid,14.017766
4,eurusd_dukascopy_ask_202510.parquet,202510,ask,13.901355
15,eurusd_dukascopy_bid_202510.parquet,202510,bid,13.918950



Month coverage (last 24):


,yyyymm,n_files
0,202101,2
1,202102,2
2,202508,2
3,202509,2
4,202510,2
5,202511,2
6,202512,2
7,202601,2
8,202602,2
9,202603,2
